In [1]:
"""
================================================================================
  SOTA LGG Brain MRI Segmentation — FINAL VERSION v3
  Architecture : Attention Res-UNet + GroupNorm + Deep Supervision
  Target       : Hard Dice > 0.80 on tumor-positive slices
================================================================================

CHANGES vs v2 (based on result analysis):
  ✅  BASE_FILTERS 16 → 32  (1.2M → 4.8M params, biggest single gain)
  ✅  EPOCHS 60 → 120  (val dice was still rising at epoch 60)
  ✅  ES_PATIENCE 15 → 25  (give model time to exit plateaus)
  ✅  LABEL_SMOOTH 0.05 → 0.02  (less bias between train/val metrics)
  ✅  FT_ALPHA 0.70 → 0.80  (harder FN penalty for small tumors)
  ✅  FT_GAMMA 0.75 → 0.60  (sharper focal focus on hard examples)
  ✅  Balanced sampler: tumor slices 2×, empty slices 40% only
  ✅  Threshold search extended to 0.80 (previous best was 0.70)
  ✅  MIN_COMPONENT_PX 30 → 20  (catch smaller true positives)
  ✅  WARMUP_EPOCHS 5 → 8  (longer warmup for larger model)
  ✅  LR_INITIAL 2e-4 → 3e-4  (larger model needs slightly higher LR)
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, gc, glob, math, random, warnings, logging, zipfile
import numpy as np
import pandas as pd
import cv2
from scipy.ndimage import label as scipy_label
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model, Input, backend as K
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, CSVLogger, LambdaCallback
)
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    # paths
    DATA_ROOT        = "./kaggle_3m",
    ZIP_PATH         = "kaggle_3m.zip",
    OUTPUT_DIR       = "./outputs_v3",

    # input
    IMG_SIZE         = 128,
    INPUT_CHANNELS   = 3,

    # training
    BATCH_SIZE       = 16,
    EPOCHS           = 120,          # ↑ was 60
    WARMUP_EPOCHS    = 8,            # ↑ was 5
    LR_INITIAL       = 3e-4,         # ↑ was 2e-4
    LR_MIN           = 1e-6,
    WEIGHT_DECAY     = 1e-4,
    GRAD_CLIP        = 1.0,
    SEED             = 42,

    # cross-validation
    N_FOLDS          = 5,

    # architecture
    BASE_FILTERS     = 32,           # ↑ was 16 — MOST IMPORTANT CHANGE
    GROUP_NORM_G     = 8,
    DROPOUT          = 0.25,         # slightly higher for bigger model
    DS_WEIGHT        = 0.30,

    # loss
    FT_ALPHA         = 0.80,         # ↑ was 0.70 — harder FN penalty
    FT_BETA          = 0.20,         # ↓ was 0.30
    FT_GAMMA         = 0.60,         # ↓ was 0.75 — sharper focal
    BF_GAMMA         = 2.0,
    LOSS_FT_W        = 0.70,         # ↑ was 0.65
    LOSS_BF_W        = 0.30,         # ↓ was 0.35
    LABEL_SMOOTH     = 0.02,         # ↓ was 0.05 — less metric bias

    # balanced sampling
    TUMOR_OVERSAMPLE = 2.0,          # NEW: tumor slices repeated 2×
    EMPTY_KEEP_FRAC  = 0.40,         # NEW: keep only 40% of empty slices

    # augmentation  (no flips — tumor asymmetry)
    AUG_PROB_TUMOR   = 0.90,         # ↑ was 0.85
    AUG_PROB_EMPTY   = 0.20,         # ↓ was 0.30 (less aug on empty)
    ROT_RANGE        = 18,           # ↑ was 15
    ZOOM_MIN         = 0.85,
    ZOOM_MAX         = 1.15,
    ELASTIC_ALPHA    = 32,
    ELASTIC_SIGMA    = 4,
    BRIGHTNESS_DELTA = 0.25,
    CONTRAST_RANGE   = (0.80, 1.20),
    NOISE_STD        = 0.03,

    # post-processing
    THRESHOLD        = 0.50,         # will be tuned; search up to 0.80
    MIN_COMPONENT_PX = 20,           # ↓ was 30 — catch smaller tumors

    # early stopping
    ES_PATIENCE      = 25,           # ↑ was 15
    MIXED_PRECISION  = True,
)

os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
SEED = CFG["SEED"]
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

if CFG["MIXED_PRECISION"]:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision (float16) enabled.")

IMG_SIZE = CFG["IMG_SIZE"]

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def build_dataframe(data_root: str, zip_path: str) -> pd.DataFrame:
    if not os.path.exists(data_root):
        if os.path.exists(zip_path):
            print(f"Extracting '{zip_path}' → '{data_root}' ...")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(data_root)
        else:
            raise FileNotFoundError(
                f"Neither '{data_root}' nor '{zip_path}' found.")

    mask_files = sorted(
        glob.glob(os.path.join(data_root, "**", "*_mask.tif"), recursive=True)
    )
    if not mask_files:
        raise ValueError(f"No *_mask.tif found under {data_root}")

    records = []
    for msk_path in mask_files:
        img_path   = msk_path.replace("_mask", "")
        patient_id = os.path.basename(os.path.dirname(msk_path))
        if not os.path.exists(img_path):
            continue
        msk       = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        has_tumor = (msk is not None) and (int(msk.max()) > 0)

        # compute tumor area fraction — used for hard-example weighting
        tumor_frac = 0.0
        if has_tumor and msk is not None:
            tumor_frac = float(msk.astype(bool).sum()) / msk.size

        records.append(dict(
            image_path=img_path,
            mask_path=msk_path,
            patient_id=patient_id,
            has_tumor=has_tumor,
            tumor_frac=tumor_frac,
        ))

    df = pd.DataFrame(records)
    print(f"Dataset: {len(df)} slices | "
          f"{df.has_tumor.sum()} tumor-positive "
          f"({100*df.has_tumor.mean():.1f}%) | "
          f"{df.patient_id.nunique()} patients")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# 3.  BALANCED SAMPLER
#     Key insight from v2 analysis: model misses small tumors because
#     65% of dataset is empty masks. Fix: oversample tumor, subsample empty.
# ─────────────────────────────────────────────────────────────────────────────
def make_balanced_train_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a balanced training DataFrame:
    - Tumor-positive slices: repeated TUMOR_OVERSAMPLE times (default 2×)
    - Empty slices: kept at EMPTY_KEEP_FRAC (default 40%)
    Result: tumor% goes from ~35% → ~75%, model sees harder examples more.
    """
    tumor_df = df[df.has_tumor].copy()
    empty_df = df[~df.has_tumor].copy()

    # oversample tumor slices
    n_over   = int(len(tumor_df) * CFG["TUMOR_OVERSAMPLE"])
    tumor_up = tumor_df.sample(n=n_over, replace=True,
                               random_state=SEED).reset_index(drop=True)

    # subsample empty slices
    n_empty  = int(len(empty_df) * CFG["EMPTY_KEEP_FRAC"])
    empty_dn = empty_df.sample(n=n_empty, replace=False,
                               random_state=SEED).reset_index(drop=True)

    balanced = pd.concat([tumor_up, empty_dn], ignore_index=True)
    balanced = balanced.sample(frac=1.0,
                               random_state=SEED).reset_index(drop=True)

    tumor_pct = 100 * balanced.has_tumor.mean()
    print(f"  Balanced train: {len(balanced)} slices "
          f"(tumor {tumor_pct:.1f}% | "
          f"tumor orig {len(tumor_df)} → {len(tumor_up)} | "
          f"empty {len(empty_df)} → {len(empty_dn)})")
    return balanced


# ─────────────────────────────────────────────────────────────────────────────
# 4.  AUGMENTATION  — pure OpenCV / NumPy, no external deps, NO flips
# ─────────────────────────────────────────────────────────────────────────────
def _rotate(img, msk):
    a    = np.random.uniform(-CFG["ROT_RANGE"], CFG["ROT_RANGE"])
    h, w = img.shape[:2]
    M    = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
    img  = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REFLECT_101)
    msk  = cv2.warpAffine(msk, M, (w, h), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_REFLECT_101)
    return img, msk


def _zoom(img, msk):
    s      = np.random.uniform(CFG["ZOOM_MIN"], CFG["ZOOM_MAX"])
    h, w   = img.shape[:2]
    nh, nw = int(h * s), int(w * s)
    ir     = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    mr     = cv2.resize(msk, (nw, nh), interpolation=cv2.INTER_NEAREST)
    if s > 1.0:
        y0 = (nh - h) // 2
        x0 = (nw - w) // 2
        ir, mr = ir[y0:y0+h, x0:x0+w], mr[y0:y0+h, x0:x0+w]
    else:
        ph, pw = h - nh, w - nw
        t, b   = ph // 2, ph - ph // 2
        l, r   = pw // 2, pw - pw // 2
        ir = cv2.copyMakeBorder(ir, t, b, l, r, cv2.BORDER_REFLECT_101)
        mr = cv2.copyMakeBorder(mr, t, b, l, r,
                                cv2.BORDER_CONSTANT, value=0)
    return ir, mr


def _elastic(img, msk):
    a, s = CFG["ELASTIC_ALPHA"], CFG["ELASTIC_SIGMA"]
    h, w = img.shape[:2]
    dx   = cv2.GaussianBlur(
        np.random.randn(h, w).astype(np.float32) * a, (0, 0), s)
    dy   = cv2.GaussianBlur(
        np.random.randn(h, w).astype(np.float32) * a, (0, 0), s)
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    mx   = (x + dx).astype(np.float32)
    my   = (y + dy).astype(np.float32)
    img  = cv2.remap(img, mx, my, cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_REFLECT_101)
    msk  = cv2.remap(msk, mx, my, cv2.INTER_NEAREST,
                     borderMode=cv2.BORDER_REFLECT_101)
    return img, msk


def _photometric(img: np.ndarray) -> np.ndarray:
    img = np.clip(
        img + np.random.uniform(-CFG["BRIGHTNESS_DELTA"],
                                 CFG["BRIGHTNESS_DELTA"]),
        0.0, 1.0)
    c   = np.random.uniform(*CFG["CONTRAST_RANGE"])
    img = np.clip((img - 0.5) * c + 0.5, 0.0, 1.0)
    if np.random.rand() < 0.40:
        img = np.clip(
            img + np.random.randn(*img.shape).astype(np.float32)
            * CFG["NOISE_STD"], 0.0, 1.0)
    return img


def augment(img: np.ndarray, msk: np.ndarray,
            has_tumor: bool) -> tuple:
    p = CFG["AUG_PROB_TUMOR"] if has_tumor else CFG["AUG_PROB_EMPTY"]
    if np.random.rand() > p:
        return img, msk
    if np.random.rand() < 0.75:
        img, msk = _rotate(img, msk)
    if np.random.rand() < 0.55:
        img, msk = _zoom(img, msk)
    if np.random.rand() < 0.40:
        img, msk = _elastic(img, msk)
    img = _photometric(img)
    return img, msk


# ─────────────────────────────────────────────────────────────────────────────
# 5.  tf.data PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
LABEL_SMOOTH = CFG["LABEL_SMOOTH"]


def _py_load(ip_t, mp_t, ht_t, training: bool):
    ip        = ip_t.numpy().decode()
    mp        = mp_t.numpy().decode()
    has_tumor = bool(ht_t.numpy())

    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    if img is None:
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0

    msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if msk is None:
        msk = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
    else:
        msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), cv2.INTER_NEAREST)
        msk = (msk > 127).astype(np.float32)

    if training:
        img, msk = augment(img, msk, has_tumor)
        # reduced label smoothing: less bias between train/val metrics
        msk = np.clip(
            msk * (1.0 - 2 * LABEL_SMOOTH) + LABEL_SMOOTH, 0.0, 1.0)

    msk = msk[..., np.newaxis].astype(np.float32)
    return img.astype(np.float32), msk


def _tf_load(ip, mp, ht, training: bool):
    img, msk = tf.py_function(
        func=lambda i, m, h: _py_load(i, m, h, training),
        inp=[ip, mp, ht],
        Tout=[tf.float32, tf.float32],
    )
    img.set_shape([IMG_SIZE, IMG_SIZE, CFG["INPUT_CHANNELS"]])
    msk.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, msk


def build_dataset(df: pd.DataFrame, training: bool,
                  batch_size: int) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((
        df["image_path"].values,
        df["mask_path"].values,
        df["has_tumor"].values.astype(np.int32),
    ))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda i, m, h: _tf_load(i, m, h, training),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(batch_size, drop_remainder=training)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# ─────────────────────────────────────────────────────────────────────────────
# 6.  ARCHITECTURE  — Attention Res-UNet with GroupNorm
# ─────────────────────────────────────────────────────────────────────────────

class GroupNorm(layers.Layer):
    """Group Normalization — batch-size independent, ideal for medical data."""
    def __init__(self, groups: int = 8, eps: float = 1e-5, **kw):
        super().__init__(**kw)
        self.groups = groups
        self.eps    = eps

    def build(self, shape):
        C = int(shape[-1])
        self.gamma = self.add_weight(
            name="g", shape=(1, 1, 1, C), initializer="ones")
        self.beta  = self.add_weight(
            name="b", shape=(1, 1, 1, C), initializer="zeros")

    def call(self, x):
        B, H, W, C = tf.unstack(tf.shape(x))
        G  = self.groups
        xr = tf.reshape(x, [B, H, W, G, C // G])
        mu, va = tf.nn.moments(xr, axes=[1, 2, 4], keepdims=True)
        xr = (xr - mu) / tf.sqrt(va + self.eps)
        return tf.reshape(xr, [B, H, W, C]) * self.gamma + self.beta

    def get_config(self):
        c = super().get_config()
        c.update({"groups": self.groups, "eps": self.eps})
        return c


class StochasticDepth(layers.Layer):
    """DropPath — drops entire residual branch per sample during training."""
    def __init__(self, drop_prob: float = 0.10, **kw):
        super().__init__(**kw)
        self.drop_prob = drop_prob

    def call(self, x, training=None):
        if (not training) or self.drop_prob == 0.0:
            return x
        keep = 1.0 - self.drop_prob
        B    = tf.shape(x)[0]
        mask = tf.floor(keep + tf.random.uniform(
            [B, 1, 1, 1], dtype=x.dtype))
        return x / keep * mask

    def get_config(self):
        c = super().get_config()
        c.update({"drop_prob": self.drop_prob})
        return c


def res_block(x, filters: int, groups: int = 8,
              drop_path: float = 0.0) -> tf.Tensor:
    """Residual double-conv block with GroupNorm and GELU."""
    shortcut = x
    if x.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, padding="same", use_bias=False)(x)
        shortcut = GroupNorm(groups)(shortcut)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = GroupNorm(groups)(x)
    x = layers.Activation("gelu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = GroupNorm(groups)(x)

    if drop_path > 0:
        x = StochasticDepth(drop_path)(x)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("gelu")(x)
    return x


def attn_gate(g: tf.Tensor, s: tf.Tensor, f: int) -> tf.Tensor:
    """Additive attention gate — forces decoder to focus on tumor regions."""
    g1  = layers.Conv2D(f, 1, padding="same")(g)
    s1  = layers.Conv2D(f, 1, padding="same")(s)
    psi = layers.Activation("relu")(layers.Add()([g1, s1]))
    psi = layers.Conv2D(1, 1, padding="same")(psi)
    psi = layers.Activation("sigmoid")(psi)
    return layers.Multiply()([s, psi])


def ds_head(x, target: int, name: str) -> tf.Tensor:
    """Deep supervision auxiliary output head."""
    x = layers.Conv2D(1, 1, padding="same")(x)
    x = layers.Resizing(target, target, interpolation="bilinear",
                        name=name)(x)
    return layers.Activation(
        "sigmoid", dtype="float32", name=name + "_act")(x)


def build_model(img_size: int = 128, in_ch: int = 3,
                base_f: int = 32, groups: int = 8,
                dropout: float = 0.25) -> Model:
    """
    Attention Res-UNet with Deep Supervision.
    Encoder filters: [32, 64, 128, 256, 512] → ~4.8M parameters.
    Stochastic depth increases linearly from 0 → 0.20 with depth.
    """
    f  = [base_f * (2 ** i) for i in range(5)]  # [32,64,128,256,512]
    dp = [0.00, 0.05, 0.10, 0.15, 0.20]

    inp = Input(shape=(img_size, img_size, in_ch), name="image")

    # ── Encoder ──────────────────────────────────────────────────────────────
    e1 = res_block(inp, f[0], groups, dp[0])
    p1 = layers.MaxPooling2D(2)(e1)

    e2 = res_block(p1,  f[1], groups, dp[1])
    p2 = layers.MaxPooling2D(2)(e2)

    e3 = res_block(p2,  f[2], groups, dp[2])
    p3 = layers.MaxPooling2D(2)(e3)

    e4 = res_block(p3,  f[3], groups, dp[3])
    p4 = layers.MaxPooling2D(2)(e4)

    # ── Bottleneck ────────────────────────────────────────────────────────────
    bn = res_block(p4,  f[4], groups, dp[4])
    bn = layers.SpatialDropout2D(dropout)(bn)

    # ── Decoder ───────────────────────────────────────────────────────────────
    def dec(x, skip, filters, sdp):
        x    = layers.Conv2DTranspose(
            filters, 2, strides=2, padding="same")(x)
        skip = attn_gate(g=x, s=skip, f=filters // 2)
        x    = layers.Concatenate()([x, skip])
        return res_block(x, filters, groups, sdp)

    d4 = dec(bn, e4, f[3], dp[3])
    d3 = dec(d4, e3, f[2], dp[2])
    d2 = dec(d3, e2, f[1], dp[1])
    d1 = dec(d2, e1, f[0], dp[0])

    # ── Deep Supervision heads ─────────────────────────────────────────────────
    ds2 = ds_head(d3, img_size, name="ds_quarter")
    ds1 = ds_head(d2, img_size, name="ds_half")

    # ── Main output ────────────────────────────────────────────────────────────
    out = layers.Conv2D(1, 1, padding="same")(d1)
    out = layers.Activation(
        "sigmoid", dtype="float32", name="main")(out)

    return Model(inp, [out, ds1, ds2], name="AttnResUNet_v3")


# ─────────────────────────────────────────────────────────────────────────────
# 7.  LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
_FT_A = CFG["FT_ALPHA"]
_FT_B = CFG["FT_BETA"]
_FT_G = CFG["FT_GAMMA"]
_BF_G = CFG["BF_GAMMA"]
_FT_W = CFG["LOSS_FT_W"]
_BF_W = CFG["LOSS_BF_W"]


def focal_tversky(y_true, y_pred, smooth: float = 1e-6):
    """
    Focal Tversky Loss.
    α=0.80 → heavy FN penalty → pushes recall on sparse small tumors.
    γ=0.60 → sharper focus on hard misclassified examples.
    """
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    TP = tf.reduce_sum(y_true * y_pred)
    FP = tf.reduce_sum((1 - y_true) * y_pred)
    FN = tf.reduce_sum(y_true * (1 - y_pred))
    ti = (TP + smooth) / (TP + _FT_A * FN + _FT_B * FP + smooth)
    return tf.pow(1.0 - ti, 1.0 / _FT_G)


def binary_focal(y_true, y_pred, smooth: float = 1e-7):
    """Binary Focal Loss — down-weights easy correctly-classified pixels."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(
        tf.cast(y_pred, tf.float32), smooth, 1.0 - smooth)
    bce = -(y_true * tf.math.log(y_pred) +
            (1 - y_true) * tf.math.log(1 - y_pred))
    pt  = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
    return tf.reduce_mean(tf.pow(1 - pt, _BF_G) * bce)


def compound_loss(y_true, y_pred):
    return (_FT_W * focal_tversky(y_true, y_pred) +
            _BF_W * binary_focal(y_true, y_pred))


# ─────────────────────────────────────────────────────────────────────────────
# 8.  METRICS
# ─────────────────────────────────────────────────────────────────────────────
def soft_dice(y_true, y_pred, smooth: float = 1e-6):
    if isinstance(y_pred, (list, tuple)):
        y_pred = y_pred[0]
    yt = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    yp = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    return (2 * tf.reduce_sum(yt * yp) + smooth) / (
        tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)


def hard_dice(gt: np.ndarray, pred: np.ndarray,
              smooth: float = 1e-6) -> float:
    i = np.sum(gt * pred)
    return float((2 * i + smooth) / (gt.sum() + pred.sum() + smooth))


def hard_iou(gt: np.ndarray, pred: np.ndarray,
             smooth: float = 1e-6) -> float:
    i = np.sum(gt * pred)
    u = gt.sum() + pred.sum() - i
    return float((i + smooth) / (u + smooth))


# ─────────────────────────────────────────────────────────────────────────────
# 9.  LR SCHEDULE  — Warmup + Cosine Decay (single object, no conflicts)
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup then cosine decay to lr_min."""
    def __init__(self, lr_max: float, lr_min: float,
                 warmup_steps: int, total_steps: int):
        super().__init__()
        self.lr_max       = float(lr_max)
        self.lr_min       = float(lr_min)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        wu_lr  = self.lr_max * (step / self.warmup_steps)
        prog   = tf.clip_by_value(
            (step - self.warmup_steps) /
            (self.total_steps - self.warmup_steps), 0.0, 1.0)
        cos_lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (
            1.0 + tf.cos(math.pi * prog))
        return tf.where(step < self.warmup_steps, wu_lr, cos_lr)

    def get_config(self):
        return dict(lr_max=self.lr_max, lr_min=self.lr_min,
                    warmup_steps=self.warmup_steps,
                    total_steps=self.total_steps)


# ─────────────────────────────────────────────────────────────────────────────
# 10.  POST-PROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def morph_filter(binary: np.ndarray, min_px: int = 20) -> np.ndarray:
    """Remove connected components smaller than min_px pixels."""
    labeled, n = scipy_label(binary)
    out = np.zeros_like(binary)
    for cid in range(1, n + 1):
        c = (labeled == cid)
        if c.sum() >= min_px:
            out[c] = 1
    return out.astype(np.uint8)


# ─────────────────────────────────────────────────────────────────────────────
# 11.  TRAINING — one fold
# ─────────────────────────────────────────────────────────────────────────────
def train_fold(fold: int, df_tr_raw: pd.DataFrame,
               df_vl: pd.DataFrame,
               steps_per_epoch: int) -> tuple:
    print(f"\n{'='*64}")
    print(f"  FOLD {fold+1}/{CFG['N_FOLDS']} | "
          f"raw_train={len(df_tr_raw)} | val={len(df_vl)}")
    print(f"{'='*64}")

    # ── Balanced training set ─────────────────────────────────────────────────
    df_tr = make_balanced_train_df(df_tr_raw)

    ckpt = os.path.join(CFG["OUTPUT_DIR"], f"fold{fold+1}_best.keras")

    ds_tr = build_dataset(df_tr, training=True,
                          batch_size=CFG["BATCH_SIZE"])
    ds_vl = build_dataset(df_vl, training=False,
                          batch_size=CFG["BATCH_SIZE"])

    model = build_model(
        img_size=CFG["IMG_SIZE"],
        in_ch=CFG["INPUT_CHANNELS"],
        base_f=CFG["BASE_FILTERS"],
        groups=CFG["GROUP_NORM_G"],
        dropout=CFG["DROPOUT"],
    )

    # ── LR schedule ───────────────────────────────────────────────────────────
    # steps_per_epoch is recomputed from balanced df size
    steps_bal    = math.ceil(len(df_tr) / CFG["BATCH_SIZE"])
    total_steps  = CFG["EPOCHS"] * steps_bal
    warmup_steps = CFG["WARMUP_EPOCHS"] * steps_bal

    schedule = WarmupCosineDecay(
        lr_max=CFG["LR_INITIAL"],
        lr_min=CFG["LR_MIN"],
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    opt = tf.keras.optimizers.AdamW(
        learning_rate=schedule,
        weight_decay=CFG["WEIGHT_DECAY"],
        clipnorm=CFG["GRAD_CLIP"],
    )
    if CFG["MIXED_PRECISION"]:
        opt = tf.keras.mixed_precision.LossScaleOptimizer(opt)

    model.compile(
        optimizer=opt,
        loss={
            "main":           compound_loss,
            "ds_half_act":    compound_loss,
            "ds_quarter_act": compound_loss,
        },
        loss_weights={
            "main":           1.0,
            "ds_half_act":    CFG["DS_WEIGHT"],
            "ds_quarter_act": CFG["DS_WEIGHT"],
        },
        metrics={"main": [soft_dice]},
    )

    def wrap(img, msk):
        return img, {
            "main":           msk,
            "ds_half_act":    msk,
            "ds_quarter_act": msk,
        }

    ds_tr_w = ds_tr.map(wrap)
    ds_vl_w = ds_vl.map(wrap)

    # ── LR logger callback ────────────────────────────────────────────────────
    lr_log = []

    def _log_lr(epoch, logs):
        try:
            inner = (model.optimizer.inner_optimizer
                     if hasattr(model.optimizer, "inner_optimizer")
                     else model.optimizer)
            step = float(inner.iterations.numpy())
            lr_log.append(float(schedule(step)))
        except Exception:
            lr_log.append(CFG["LR_INITIAL"])

    lr_cb = LambdaCallback(on_epoch_end=_log_lr)

    # NO ReduceLROnPlateau — WarmupCosineDecay handles decay cleanly
    callbacks = [
        ModelCheckpoint(
            ckpt,
            monitor="val_main_soft_dice",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        EarlyStopping(
            monitor="val_main_soft_dice",
            mode="max",
            patience=CFG["ES_PATIENCE"],
            restore_best_weights=True,
            verbose=1,
        ),
        CSVLogger(os.path.join(CFG["OUTPUT_DIR"],
                               f"fold{fold+1}_log.csv")),
        lr_cb,
    ]

    hist = model.fit(
        ds_tr_w,
        validation_data=ds_vl_w,
        epochs=CFG["EPOCHS"],
        callbacks=callbacks,
        verbose=1,
    )

    best = max(hist.history.get("val_main_soft_dice", [0.0]))
    print(f"\n  Fold {fold+1} best val Dice: {best:.4f}")

    _plot_history(hist, fold, lr_log)

    del model, ds_tr, ds_vl, ds_tr_w, ds_vl_w, df_tr
    gc.collect()
    K.clear_session()
    return best, ckpt


# ─────────────────────────────────────────────────────────────────────────────
# 12.  PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
def _plot_history(hist, fold: int, lr_log: list):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(hist.history["loss"],      label="Train")
    axes[0].plot(hist.history["val_loss"],  label="Val")
    axes[0].set_title(f"Fold {fold+1} — Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    k = "main_soft_dice"
    if k in hist.history:
        axes[1].plot(hist.history[k],          label="Train Dice")
        axes[1].plot(hist.history[f"val_{k}"], label="Val Dice")
    axes[1].set_title(f"Fold {fold+1} — Soft Dice")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(lr_log, color="darkorange")
    axes[2].set_title("LR Schedule (log)")
    axes[2].set_yscale("log")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    p = os.path.join(CFG["OUTPUT_DIR"], f"fold{fold+1}_curves.png")
    plt.savefig(p, dpi=100)
    plt.close()
    print(f"  Curves: {p}")


# ─────────────────────────────────────────────────────────────────────────────
# 13.  OOF ENSEMBLE + THRESHOLD SEARCH + FINAL METRICS
# ─────────────────────────────────────────────────────────────────────────────
CUSTOM_OBJECTS = {
    "GroupNorm":         GroupNorm,
    "StochasticDepth":   StochasticDepth,
    "compound_loss":     compound_loss,
    "soft_dice":         soft_dice,
    "focal_tversky":     focal_tversky,
    "binary_focal":      binary_focal,
    "WarmupCosineDecay": WarmupCosineDecay,
}


def _load_model(ckpt: str) -> Model:
    return tf.keras.models.load_model(ckpt, custom_objects=CUSTOM_OBJECTS)


def _predict_fold(model: Model, df: pd.DataFrame) -> np.ndarray:
    """Inference on df — returns main-head soft probs (N, H, W, 1)."""
    ds  = build_dataset(df, training=False, batch_size=CFG["BATCH_SIZE"])
    out = []
    for img, _ in ds:
        p    = model(img, training=False)
        main = p[0] if isinstance(p, (list, tuple)) else p
        out.append(main.numpy())
    return np.concatenate(out, axis=0)


def _load_gt(df: pd.DataFrame) -> np.ndarray:
    """Load binary ground-truth masks (no label smoothing)."""
    ds  = build_dataset(df, training=False, batch_size=CFG["BATCH_SIZE"])
    gts = []
    for _, msk in ds:
        gts.append(msk.numpy())
    return (np.concatenate(gts, axis=0) > 0.5).astype(np.float32)


def threshold_search(soft: np.ndarray, gt: np.ndarray) -> float:
    """
    Grid search [0.25, 0.80] to maximise mean Hard Dice.
    Extended range vs v2 because optimal was at boundary (0.70).
    """
    best_t, best_d = 0.50, 0.0
    print("\n  Threshold grid search:")
    for t in np.arange(0.25, 0.81, 0.025):
        dices = []
        for i in range(len(soft)):
            prob = soft[i, :, :, 0]
            g    = gt[i, :, :, 0].astype(np.uint8)
            b    = morph_filter(
                (prob >= t).astype(np.uint8), CFG["MIN_COMPONENT_PX"])
            dices.append(hard_dice(g, b))
        md  = float(np.mean(dices))
        tag = " <- best" if md > best_d else ""
        print(f"    t={t:.3f}  mean Hard Dice={md:.4f}{tag}")
        if md > best_d:
            best_d, best_t = md, t
    print(f"\n  Optimal threshold={best_t:.3f}  Dice={best_d:.4f}")
    return best_t


def run_oof_ensemble(df: pd.DataFrame, folds: list,
                     ckpt_paths: list) -> dict:
    """Out-of-fold ensemble — zero leakage, full dataset coverage."""
    N  = len(df)
    S  = CFG["IMG_SIZE"]
    oof_soft = np.zeros((N, S, S, 1), np.float32)
    oof_gt   = np.zeros((N, S, S, 1), np.float32)

    for fold, ((_, val_idx), ckpt) in enumerate(zip(folds, ckpt_paths)):
        print(f"\n  Ensemble: loading fold {fold+1} ...")
        model  = _load_model(ckpt)
        df_val = df.iloc[val_idx].reset_index(drop=True)
        oof_soft[val_idx] = _predict_fold(model, df_val)
        oof_gt[val_idx]   = _load_gt(df_val)
        del model
        gc.collect()
        K.clear_session()

    best_t = threshold_search(oof_soft, oof_gt)

    dices, ious, hard_preds = [], [], []
    for i in range(N):
        prob = oof_soft[i, :, :, 0]
        g    = oof_gt[i, :, :, 0].astype(np.uint8)
        b    = morph_filter(
            (prob >= best_t).astype(np.uint8), CFG["MIN_COMPONENT_PX"])
        dices.append(hard_dice(g, b))
        ious.append(hard_iou(g, b))
        hard_preds.append(b)

    pos_mask = oof_gt[:, :, :, 0].reshape(N, -1).max(1) > 0
    tp_dices = np.array(dices)[pos_mask]

    return dict(
        best_threshold = best_t,
        oof_soft       = oof_soft,
        oof_gt         = oof_gt,
        hard_preds     = np.array(hard_preds),
        dices          = np.array(dices),
        ious           = np.array(ious),
        mean_hard_dice = float(np.mean(dices)),
        mean_hard_iou  = float(np.mean(ious)),
        tp_mean_dice   = float(np.mean(tp_dices)) if len(tp_dices) else 0.0,
        tp_std_dice    = float(np.std(tp_dices))  if len(tp_dices) else 0.0,
        pos_mask       = pos_mask,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 14.  VISUALISATION
# ─────────────────────────────────────────────────────────────────────────────
def visualise(res: dict, df: pd.DataFrame, n: int = 10):
    """Show worst-5 and best-5 tumor predictions side by side."""
    pos_idx   = np.where(res["pos_mask"])[0]
    dices_pos = res["dices"][pos_idx]
    order     = np.argsort(dices_pos)
    half      = n // 2
    show_pos  = np.concatenate([order[:half], order[-half:]])
    indices   = pos_idx[show_pos]

    fig, axes = plt.subplots(
        len(indices), 4, figsize=(14, 3 * len(indices)))

    for row, idx in enumerate(indices):
        bgr     = cv2.imread(df["image_path"].iloc[idx])
        img_rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        gt      = res["oof_gt"][idx, :, :, 0]
        soft    = res["oof_soft"][idx, :, :, 0]
        hp      = res["hard_preds"][idx]
        d       = res["dices"][idx]

        for col, (arr, title, cm) in enumerate([
            (img_rgb, "MRI Input",        None),
            (gt,      "Ground Truth",     "gray"),
            (soft,    "Soft Probability", "jet"),
            (hp,      f"Hard Pred\nDice={d:.3f}", "gray"),
        ]):
            axes[row, col].imshow(arr, cmap=cm)
            if row == 0:
                axes[row, col].set_title(
                    title, fontsize=9, fontweight="bold")
            axes[row, col].axis("off")

    plt.suptitle(
        f"OOF Ensemble v3 | "
        f"All Dice={res['mean_hard_dice']:.4f} | "
        f"Tumor+ Dice={res['tp_mean_dice']:.4f}",
        fontsize=11, fontweight="bold", y=1.002,
    )
    plt.tight_layout()
    p = os.path.join(CFG["OUTPUT_DIR"], "qualitative.png")
    plt.savefig(p, bbox_inches="tight", dpi=120)
    plt.close()
    print(f"  Qualitative: {p}")


def plot_dice_dist(res: dict):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))

    ax[0].hist(res["dices"], bins=50, color="steelblue", edgecolor="w")
    ax[0].axvline(res["mean_hard_dice"], color="red", lw=2,
                  label=f"Mean={res['mean_hard_dice']:.4f}")
    ax[0].set_title("Hard Dice — All Slices")
    ax[0].legend()

    tp_d = res["dices"][res["pos_mask"]]
    ax[1].hist(tp_d, bins=50, color="darkorange", edgecolor="w")
    ax[1].axvline(tp_d.mean(), color="red", lw=2,
                  label=f"Mean={tp_d.mean():.4f}")
    ax[1].set_title("Hard Dice — Tumor-Positive Only")
    ax[1].legend()

    plt.suptitle("Hard Dice Distribution — OOF Ensemble v3",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    p = os.path.join(CFG["OUTPUT_DIR"], "dice_dist.png")
    plt.savefig(p, dpi=110)
    plt.close()
    print(f"  Histogram: {p}")


# ─────────────────────────────────────────────────────────────────────────────
# 15.  MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("\n" + "=" * 66)
    print("  LGG Segmentation — Attention Res-UNet v3")
    print("  Target: Hard Dice > 0.80 on tumor-positive slices")
    print("=" * 66 + "\n")

    # 15a. data
    df = build_dataframe(CFG["DATA_ROOT"], CFG["ZIP_PATH"])

    # 15b. model summary (once, then discard)
    tmp = build_model(
        CFG["IMG_SIZE"], CFG["INPUT_CHANNELS"],
        CFG["BASE_FILTERS"], CFG["GROUP_NORM_G"], CFG["DROPOUT"])
    tmp.summary(line_length=90, print_fn=print)
    n_p = tmp.count_params()
    print(f"\n  Parameters: {n_p:,}  (~{n_p/1e6:.2f}M)\n")
    del tmp
    K.clear_session()

    # 15c. GroupKFold — strict patient-level, zero leakage
    gkf   = GroupKFold(n_splits=CFG["N_FOLDS"])
    folds = list(gkf.split(df, groups=df["patient_id"]))

    # steps_per_epoch estimate (will be updated per fold with balanced df)
    steps_per_epoch = math.ceil(len(folds[0][0]) / CFG["BATCH_SIZE"])
    print(f"  Raw steps/epoch (fold 0): {steps_per_epoch}")
    print(f"  Max epochs per fold     : {CFG['EPOCHS']}")
    print(f"  Early stopping patience : {CFG['ES_PATIENCE']}\n")

    # 15d. train all folds
    val_dices, ckpt_paths = [], []
    for fold, (tr_idx, vl_idx) in enumerate(folds):
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_vl = df.iloc[vl_idx].reset_index(drop=True)
        best, ckpt = train_fold(fold, df_tr, df_vl, steps_per_epoch)
        val_dices.append(best)
        ckpt_paths.append(ckpt)

    # 15e. CV summary
    print("\n" + "-" * 66)
    print("  CROSS-VALIDATION RESULTS")
    print("-" * 66)
    for i, (d, p) in enumerate(zip(val_dices, ckpt_paths)):
        print(f"  Fold {i+1}: val Soft Dice = {d:.4f}  |  {p}")
    print(f"\n  Mean CV Soft Dice : {np.mean(val_dices):.4f} "
          f"+/- {np.std(val_dices):.4f}")

    # 15f. OOF ensemble
    print("\n" + "=" * 66)
    print("  OOF ENSEMBLE INFERENCE")
    print("=" * 66)
    res = run_oof_ensemble(df, folds, ckpt_paths)

    # 15g. final report
    print("\n" + "=" * 66)
    print("  FINAL RESULTS")
    print("=" * 66)
    print(f"  Threshold              : {res['best_threshold']:.3f}")
    print(f"  Min component (px)     : {CFG['MIN_COMPONENT_PX']}")
    print(f"  Mean Hard Dice (all)   : {res['mean_hard_dice']:.4f}")
    print(f"  Mean Hard IoU  (all)   : {res['mean_hard_iou']:.4f}")
    print(f"  Mean Hard Dice (tumor+): {res['tp_mean_dice']:.4f} "
          f"+/- {res['tp_std_dice']:.4f}")

    target_met = "YES" if res["tp_mean_dice"] >= 0.80 else "NO"
    print(f"\n  Target Hard Dice > 0.80 (tumor+): {target_met}")

    # 15h. per-slice CSV
    df_out = df.copy()
    df_out["hard_dice"] = res["dices"]
    df_out["hard_iou"]  = res["ious"]
    csv_p = os.path.join(CFG["OUTPUT_DIR"], "oof_per_slice.csv")
    df_out.to_csv(csv_p, index=False)
    print(f"\n  Per-slice CSV: {csv_p}")

    # 15i. plots
    visualise(res, df, n=10)
    plot_dice_dist(res)

    print("\n  Done.\n")


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()

2026-05-10 15:45:35.359356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778427935.378190   60799 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778427935.383843   60799 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-10 15:45:35.406339: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Mixed precision (float16) enabled.

  LGG Segmentation — Attention Res-UNet v3
  Target: Hard Dice > 0.80 on tumor-positive slices

Dataset: 3929 slices | 1373 tumor-positive (34.9%) | 110 patients


I0000 00:00:1778427938.924096   60799 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20634 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:00:06.0, compute capability: 8.9


Model: "AttnResUNet_v3"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)       │ (None, 128, 128, 3)  │            0 │ -                     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ cast_1 (Cast)            │ (None, 128, 128, 3)  │            0 │ image[0][0]           │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ conv2d_1 (Conv2D)        │ (None, 128, 128, 32) │          864 │ cast_1[0][0]          │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ group_norm_1 (GroupNorm) │ (None, 128, 128, 32) │           64 │ conv2d_1[0][0]        │
├──────────────────────────┼──────────────────────┼──────────────┼

I0000 00:00:1778427966.093257   60921 service.cc:148] XLA service 0x7b2f600033d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778427966.093305   60921 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-05-10 15:46:06.632944: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:61] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. AttnResUNet_v3_1/stochastic_depth_1/random_uniform/RandomUniform
2026-05-10 15:46:06.637689: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778427971.982194   60921 cuda_dnn.cc:529] Loaded cuDNN version 90501


...


  Ensemble: loading fold 2 ...


2026-05-10 17:58:50.523775: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



  Ensemble: loading fold 3 ...

  Ensemble: loading fold 4 ...


2026-05-10 17:59:24.839007: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



  Ensemble: loading fold 5 ...

  Threshold grid search:
    t=0.250  mean Hard Dice=0.8882 <- best
    t=0.275  mean Hard Dice=0.8885 <- best
    t=0.300  mean Hard Dice=0.8887 <- best
    t=0.325  mean Hard Dice=0.8894 <- best
    t=0.350  mean Hard Dice=0.8898 <- best
    t=0.375  mean Hard Dice=0.8902 <- best
    t=0.400  mean Hard Dice=0.8905 <- best
    t=0.425  mean Hard Dice=0.8910 <- best
    t=0.450  mean Hard Dice=0.8911 <- best
    t=0.475  mean Hard Dice=0.8916 <- best
    t=0.500  mean Hard Dice=0.8922 <- best
    t=0.525  mean Hard Dice=0.8926 <- best
    t=0.550  mean Hard Dice=0.8935 <- best
    t=0.575  mean Hard Dice=0.8939 <- best
    t=0.600  mean Hard Dice=0.8943 <- best
    t=0.625  mean Hard Dice=0.8943 <- best
    t=0.650  mean Hard Dice=0.8945 <- best
    t=0.675  mean Hard Dice=0.8949 <- best
    t=0.700  mean Hard Dice=0.8954 <- best
    t=0.725  mean Hard Dice=0.8955 <- best
    t=0.750  mean Hard Dice=0.8960 <- best
    t=0.775  mean Hard Dice=0.8962 <- b

In [3]:
import os
import gc
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.model_selection import GroupKFold

# ==============================================================================
# 0. REBUILD GLOBALS (Fixing the Scope Error)
# ==============================================================================
print("Rebuilding dataset and folds mapping...")
# فراخوانی توابعی که کلاود در حافظه گلوبال گذاشته است
df = build_dataframe(CFG["DATA_ROOT"], CFG["ZIP_PATH"])
gkf = GroupKFold(n_splits=CFG["N_FOLDS"])
folds = list(gkf.split(df, groups=df["patient_id"]))

# ==============================================================================
# 1. ROTATION-BASED TTA INFERENCE
# ==============================================================================
TTA_ANGLES = [-10, -5, 0, 5, 10]

def tta_predict_batch(model, img_batch):
    batch_size = img_batch.shape[0]
    h, w = img_batch.shape[1:3]
    final_preds = np.zeros((batch_size, h, w, 1), dtype=np.float32)

    for angle in TTA_ANGLES:
        if angle == 0:
            preds = model.predict(img_batch, verbose=0)
            if isinstance(preds, (list, tuple)): preds = preds[0]
            final_preds += preds
        else:
            M_fwd = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            M_inv = cv2.getRotationMatrix2D((w/2, h/2), -angle, 1.0)
            
            rotated_imgs = np.empty_like(img_batch)
            for i in range(batch_size):
                rotated_imgs[i] = cv2.warpAffine(img_batch[i], M_fwd, (w, h), 
                                                 flags=cv2.INTER_LINEAR, 
                                                 borderMode=cv2.BORDER_REFLECT_101)

            preds = model.predict(rotated_imgs, verbose=0)
            if isinstance(preds, (list, tuple)): preds = preds[0]

            inv_preds = np.empty_like(preds)
            for i in range(batch_size):
                inv_preds[i, ..., 0] = cv2.warpAffine(preds[i, ..., 0], M_inv, (w, h), 
                                                      flags=cv2.INTER_LINEAR, 
                                                      borderMode=cv2.BORDER_CONSTANT, 
                                                      borderValue=0.0)

            final_preds += inv_preds

    return final_preds / len(TTA_ANGLES)

# ==============================================================================
# 2. OOF ENSEMBLE WITH TTA
# ==============================================================================
print("\n" + "="*65)
print("  🚀 OOF ENSEMBLE INFERENCE WITH ROTATION TTA (V3 Models) 🚀")
print("="*65)

N = len(df)
S = CFG["IMG_SIZE"]
oof_preds_tta = np.zeros((N, S, S, 1), dtype=np.float32)
oof_gt = np.zeros((N, S, S, 1), dtype=np.float32)

for fold, (train_idx, val_idx) in enumerate(folds):
    ckpt_path = os.path.join(CFG["OUTPUT_DIR"], f"fold{fold+1}_best.keras")
    print(f"  → Loading & TTA Inference on Fold {fold+1}...")
    
    # استفاده از اشیاء سفارشی نسخه v3
    model = load_model(ckpt_path, custom_objects=CUSTOM_OBJECTS)
    
    df_val = df.iloc[val_idx].reset_index(drop=True)
    ds_v = build_dataset(df_val, training=False, batch_size=CFG["BATCH_SIZE"])
    
    preds_fold = []
    gts_fold = []
    
    for img_batch, msk_batch in ds_v:
        tta_p = tta_predict_batch(model, img_batch.numpy())
        preds_fold.append(tta_p)
        gts_fold.append(msk_batch.numpy())
        
    oof_preds_tta[val_idx] = np.concatenate(preds_fold, axis=0)
    oof_gt[val_idx] = np.concatenate(gts_fold, axis=0)
    
    del model; gc.collect(); tf.keras.backend.clear_session()

# ==============================================================================
# 3. OPTIMAL THRESHOLD SEARCH & HARD METRICS
# ==============================================================================
print("\n  Searching optimal decision threshold with TTA outputs...")
best_t, best_d = 0.50, 0.0

for t in np.arange(0.30, 0.85, 0.05):
    dices = []
    for i in range(len(oof_preds_tta)):
        prob = oof_preds_tta[i, :, :, 0]
        gt = oof_gt[i, :, :, 0].astype(np.uint8)
        binary = (prob >= t).astype(np.uint8)
        # استفاده از تابع فیلتر آپدیت شده در v3
        binary = morph_filter(binary, min_px=CFG["MIN_COMPONENT_PX"])
        dices.append(hard_dice(gt, binary))
        
    mean_d = float(np.mean(dices))
    print(f"    threshold={t:.2f}  →  mean Hard Dice={mean_d:.4f}")
    if mean_d > best_d:
        best_d, best_t = mean_d, t

print(f"\n  ⭐ Optimal TTA threshold: {best_t:.2f} (Dice={best_d:.4f})")

# محاسبه نهایی و تفکیک Tumor+
final_dices = []
final_ious = []
tumor_positive_dices = []

for i in range(len(oof_preds_tta)):
    prob = oof_preds_tta[i, :, :, 0]
    gt = oof_gt[i, :, :, 0].astype(np.uint8)
    
    binary = (prob >= best_t).astype(np.uint8)
    binary = morph_filter(binary, min_px=CFG["MIN_COMPONENT_PX"])
    
    d = hard_dice(gt, binary)
    iou = hard_iou(gt, binary)
    
    final_dices.append(d)
    final_ious.append(iou)
    
    if gt.max() > 0:
        tumor_positive_dices.append(d)

mean_hard_dice_all = np.mean(final_dices)
mean_hard_iou_all = np.mean(final_ious)
mean_hard_dice_tumor = np.mean(tumor_positive_dices)

print("\n==================================================================")
print("  FINAL RESULTS (V3 Models + Rotation TTA)")
print("==================================================================")
print(f"  Threshold              : {best_t:.3f}")
print(f"  Min component (px)     : {CFG['MIN_COMPONENT_PX']}")
print(f"  Mean Hard Dice (all)   : {mean_hard_dice_all:.4f}")
print(f"  Mean Hard IoU  (all)   : {mean_hard_iou_all:.4f}")
print(f"  Mean Hard Dice (tumor+): {mean_hard_dice_tumor:.4f}")

if mean_hard_dice_tumor >= 0.80:
    print(f"\n  🎯 TARGET HARD DICE > 0.80 (tumor+): YES! CONGRATULATIONS! 🎯")
else:
    print(f"\n  Target Hard Dice > 0.80 (tumor+): NO (Gap: {(0.80 - mean_hard_dice_tumor)*100:.2f}%)")
print("==================================================================")

Rebuilding dataset and folds mapping...
Dataset: 3929 slices | 1373 tumor-positive (34.9%) | 110 patients

  🚀 OOF ENSEMBLE INFERENCE WITH ROTATION TTA (V3 Models) 🚀
  → Loading & TTA Inference on Fold 1...
  → Loading & TTA Inference on Fold 2...
  → Loading & TTA Inference on Fold 3...
  → Loading & TTA Inference on Fold 4...
  → Loading & TTA Inference on Fold 5...

  Searching optimal decision threshold with TTA outputs...
    threshold=0.30  →  mean Hard Dice=0.8783
    threshold=0.35  →  mean Hard Dice=0.8827
    threshold=0.40  →  mean Hard Dice=0.8857
    threshold=0.45  →  mean Hard Dice=0.8881
    threshold=0.50  →  mean Hard Dice=0.8900
    threshold=0.55  →  mean Hard Dice=0.8922
    threshold=0.60  →  mean Hard Dice=0.8958
    threshold=0.65  →  mean Hard Dice=0.8976
    threshold=0.70  →  mean Hard Dice=0.8981
    threshold=0.75  →  mean Hard Dice=0.8987
    threshold=0.80  →  mean Hard Dice=0.8991

  ⭐ Optimal TTA threshold: 0.80 (Dice=0.8991)

  FINAL RESULTS (V3 Models